# 04 — Results summary

Reads every results CSV from `reports/` and builds the comparison table and figures for the report and the presentation.

**This is the notebook to demo live** — it runs in seconds because it only reads results that already exist.

In [1]:
import pandas as pd
from tabpfn_nids import config
from tabpfn_nids.evaluation import load_results, summarize_paired, wilcoxon_test

pd.set_option('display.width', 200, 'display.max_columns', 40)

## 1. Every run on record

Each row carries its own provenance: seed, hardware, library versions, checkpoint and git commit.

In [2]:
rows = load_results('baseline')
df = pd.DataFrame(rows)
cols = ['seed','context_rows','test_rows','n_features','n_estimators',
        'accuracy','precision','recall','f1_score','roc_auc','runtime_seconds']
df[cols].astype({c: float for c in cols[5:]}).round(4)

,seed,context_rows,test_rows,n_features,n_estimators,accuracy,precision,recall,f1_score,roc_auc,runtime_seconds
0,42,10000,5000,122,2,0.7452,0.9159,0.6082,0.7310,0.9515,542.13
1,123,10000,5000,122,2,0.7632,0.9227,0.6374,0.7539,0.9555,172.72
2,2024,10000,5000,122,2,0.7874,0.9655,0.6497,0.7767,0.9595,172.47


## 2. Mean and spread across seeds

Reported as mean ± std rather than with a significance test. The Wilcoxon signed-rank test cannot reach p < 0.05 with three seeds — its smallest attainable two-sided p at n=3 is 0.25 — so a p-value here would be uninformative by construction.

In [3]:
metrics = ['accuracy','precision','recall','f1_score','roc_auc']
summary = df[metrics].astype(float).agg(['mean','std']).T.round(4)
summary.columns = ['mean','std']
summary

,mean,std
accuracy,0.7653,0.0212
precision,0.9347,0.0269
recall,0.6318,0.0213
f1_score,0.7539,0.0229
roc_auc,0.9555,0.0040


### The noise floor

The F1 standard deviation across seeds is the threshold an enhancement has to clear before its delta means anything.

In [4]:
f1_std = df['f1_score'].astype(float).std()
print(f'F1 std across seeds: {f1_std:.4f}  ({100*f1_std:.2f} pp)')
print(f'A single-seed delta below ~{100*f1_std:.1f} pp is not evidence of an effect.')

F1 std across seeds: 0.0229  (2.29 pp)
A single-seed delta below ~2.3 pp is not evidence of an effect.


## 3. Provenance

What produced these numbers.

In [5]:
first = rows[0]
for k in ('hardware','device','tabpfn_version','checkpoint',
          'torch_version','python_version','git_commit'):
    print(f'  {k:<18} {first.get(k)}')

  hardware           macOS-26.6.2-arm64-arm-64bit (arm64, 8 cores)
  device             mps
  tabpfn_version     8.5.0
  checkpoint         tabpfn-v2-classifier.ckpt
  torch_version      2.13.0
  python_version     3.11.9
  git_commit         3d9223a


## 4. Combined comparison table

Run `python scripts/build_comparison_table.py` to regenerate `reports/tables/comparison_table.{csv,md}` across every experiment.

In [6]:
table = config.TABLES_DIR / 'comparison_table.md'
print(table.read_text() if table.exists() else 'Run scripts/build_comparison_table.py first')

# Results comparison

Mean +/- standard deviation across seeds. Standard deviation is 0.000 where only one seed was run.

| Metric | baseline |
|---|---|
| accuracy | 0.7653 ± 0.0212 |
| precision | 0.9347 ± 0.0269 |
| recall | 0.6318 ± 0.0213 |
| f1_score | 0.7539 ± 0.0229 |
| roc_auc | 0.9555 ± 0.0040 |

| Setting | baseline |
|---|---|
| seeds | 3 |
| context rows | 10000 |
| test rows | 5000 |
| features | 122 |
| chunks | - |

